# Fuentes oficiales: censo del DANE y capas de la IDESC (E7)

Después de las [variables nuevas](variables_e7.ipynb) (mercado alrededor, servicios de OpenStreetMap y
texto), el modelo quedó en ~10,7% de error mediano. OpenStreetMap es desigual entre ciudades (Cali, la ciudad
del piloto, está floja), así que aquí se prueban dos fuentes oficiales:

| Fuente | Cobertura | Variables |
|---|---|---|
| **Censo 2018 del DANE** por manzana (`src/censo_dane.py`) | **Todo el país** | Alrededor de cada aviso (manzanas a menos de 250 m): estrato medio de las viviendas, % de residentes con educación superior y con posgrado, % de apartamentos, % de viviendas desocupadas, % con internet y con gas, personas por hectárea, viviendas e **índice de pobreza multidimensional** (IPM) |
| **Capas de la IDESC** (Alcaldía) (`src/capas_cali.py`) | **Solo Cali** | Distancia a estaciones del MIO y paradas a 500 m, **homicidios a 500 m (2023–2025)**, ruido de día y de noche (dB), amenaza de inundación, barrio precario, densidad de árboles, distancia a colegios, universidades, salud y seguridad, centralidad y tratamiento urbanístico del POT, distancia a vías arterias y colectoras |

Ninguna de las dos usa precios, así que no hay riesgo de filtrar la respuesta.

**Cómo se mide:** LightGBM con la misma partición por inmueble que los otros notebooks. Se compara cada
variante con la **configuración actual** en todo el país y **en Cali** (las capas de la IDESC solo pueden
ayudar ahí), con intervalos de confianza por *bootstrap*.

In [1]:
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
from src import modelo_precio as mp

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 140)
AZUL, GRIS, TINTA, TINTA_2 = "#2a78d6", "#a8a69f", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#d6d4ce",
    "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2, "text.color": TINTA,
    "axes.grid": True, "grid.color": "#e8e6e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10.5, "legend.frameon": False,
})
pct = lambda x: f"{x:.1%}".replace(".", ",")
mil = lambda x: f"{x:,.0f}".replace(",", ".")
pp = lambda v: f"{v * 100:+.2f} pts".replace(".", ",")

## 1. Datos y variables

In [2]:
a, _ = mp.limpiar(mp.cargar())
X, TE, _ = mp.variables(a)
y, grupo = a["lpm2"].to_numpy(), mp.grupos(a)
entrenamiento, prueba = mp.particion_grupos(grupo)
ajuste, validacion = mp.subparticion(entrenamiento, grupo)
objetivo = np.concatenate([validacion, prueba])
t0 = time.time()
G = mp.variables_nuevas(a, y, ajuste, objetivo, grupo, grupos=("mercado", "servicios", "texto", "censo", "cali"))
es_cali = a["ciudad"].eq("Cali").to_numpy()
print(f"Variables calculadas en {time.time() - t0:.0f} s · avisos de Cali: {mil(es_cali.sum())} "
      f"({mil(es_cali[prueba].sum())} en la prueba)")
cob = pd.DataFrame({
    "variables": {g: G[g].shape[1] for g in ("censo", "cali")},
    "con dato (todo el país)": {g: G[g].notna().any(axis=1).mean() for g in ("censo", "cali")},
    "con dato (Cali)": {g: G[g][es_cali].notna().any(axis=1).mean() for g in ("censo", "cali")},
})
cob.style.format({"con dato (todo el país)": pct, "con dato (Cali)": pct})

Variables calculadas en 68 s · avisos de Cali: 9.259 (1.741 en la prueba)


,variables,con dato (todo el país),con dato (Cali)
censo,10,"91,0%","88,5%"
cali,16,"7,6%","100,0%"


### Cómo se relacionan con el precio

Correlación de rangos con el log del precio/m² (en Cali para las capas de la IDESC). Solo orientativa.

In [3]:
rho = lambda D, mask: D[mask].corrwith(pd.Series(y[mask], index=D.index[mask]), method="spearman")
tabla = pd.concat([
    pd.DataFrame({"fuente": "DANE (todo el país)", "correlación": rho(G["censo"], np.ones(len(a), bool)),
                  "en Cali": rho(G["censo"], es_cali)}),
    pd.DataFrame({"fuente": "IDESC (Cali)", "correlación": np.nan,
                  "en Cali": rho(G["cali"].select_dtypes("number"), es_cali)}),
])
tabla.sort_values("en Cali", key=lambda s: -s.abs()).round(2)

,fuente,correlación,en Cali
censo_pct_posgrado,DANE (todo el país),0.44,0.43
censo_estrato_medio,DANE (todo el país),0.40,0.36
censo_personas_ha,DANE (todo el país),-0.12,-0.33
cali_homicidios_500m_3a,IDESC (Cali),NaN,-0.32
censo_pct_superior,DANE (todo el país),0.38,0.31
censo_pct_internet,DANE (todo el país),0.37,0.29
censo_viviendas,DANE (todo el país),-0.04,-0.29
cali_n_paradas_mio_500m,IDESC (Cali),NaN,-0.27
cali_dist_mio_estacion_km,IDESC (Cali),NaN,0.26
cali_densidad_arborea,IDESC (Cali),NaN,-0.24


## 2. ¿Mejoran el modelo?

In [4]:
ACTUAL = pd.concat([X, G["mercado"], G["servicios"], G["texto"]], axis=1)
VARIANTES = {
    "Actual (base + mercado + servicios + texto)": ACTUAL,
    "Actual + censo DANE": pd.concat([ACTUAL, G["censo"]], axis=1),
    "Actual + capas IDESC": pd.concat([ACTUAL, G["cali"]], axis=1),
    "Actual + censo + IDESC": pd.concat([ACTUAL, G["censo"], G["cali"]], axis=1),
    "Censo en lugar de OSM": pd.concat([X, G["mercado"], G["texto"], G["censo"]], axis=1),
}
pred = {k: mp.lightgbm(Xb, TE, y, ajuste, validacion, prueba) for k, Xb in VARIANTES.items()}
errores = {k: mp.error(p, y[prueba]) for k, p in pred.items()}

rng = np.random.default_rng(mp.SEMILLA)
cali_te = es_cali[prueba]


def comparar(mascara):
    n = mascara.sum()
    muestras = rng.integers(0, n, size=(1000, n))
    ref = np.median(errores[next(iter(errores))][mascara][muestras], axis=1)
    filas = {}
    for k, e in errores.items():
        dif = np.median(e[mascara][muestras], axis=1) - ref
        lo, hi = np.percentile(dif, [2.5, 97.5])
        filas[k] = mp.metricas(e[mascara]) | {"frente al actual": np.median(dif),
                                             "IC 95%": "—" if k == next(iter(errores)) else f"{pp(lo)} a {pp(hi)}"}
    return pd.DataFrame(filas).T


FORMATO = {"MdAPE": pct, "±10%": pct, "±20%": pct, "avisos": mil, "frente al actual": pp}
display(comparar(np.ones(len(prueba), bool)).style.format(FORMATO).set_caption("Todo el país"))
display(comparar(cali_te).style.format(FORMATO).set_caption("Solo Cali"))

,MdAPE,±10%,±20%,avisos,frente al actual,IC 95%
Actual (base + mercado + servicios + texto),"10,7%","47,4%","75,0%",24.531,"+0,00 pts",—
Actual + censo DANE,"10,8%","47,2%","74,9%",24.531,"+0,00 pts","-0,12 pts a +0,11 pts"
Actual + capas IDESC,"10,8%","47,2%","74,8%",24.531,"+0,03 pts","-0,08 pts a +0,15 pts"
Actual + censo + IDESC,"10,7%","47,2%","75,0%",24.531,"-0,01 pts","-0,13 pts a +0,10 pts"
Censo en lugar de OSM,"10,7%","47,2%","75,0%",24.531,"-0,05 pts","-0,17 pts a +0,09 pts"


,MdAPE,±10%,±20%,avisos,frente al actual,IC 95%
Actual (base + mercado + servicios + texto),"11,3%","45,4%","74,2%",1.741,"+0,00 pts",—
Actual + censo DANE,"11,4%","44,9%","73,5%",1.741,"+0,10 pts","-0,37 pts a +0,62 pts"
Actual + capas IDESC,"11,6%","44,5%","72,6%",1.741,"+0,30 pts","-0,17 pts a +0,73 pts"
Actual + censo + IDESC,"11,7%","43,3%","72,7%",1.741,"+0,33 pts","-0,18 pts a +0,83 pts"
Censo en lugar de OSM,"11,4%","43,9%","73,9%",1.741,"+0,02 pts","-0,48 pts a +0,53 pts"


### ¿Y donde hay pocos comparables?

Los precios de los vecinos ya resumen buena parte de lo que dicen el censo o la IDESC. Donde hay **pocos
avisos cerca** (el 20% de la prueba con los 10 vecinos del mismo tipo más lejanos), las fuentes oficiales
podrían aportar más.

In [5]:
lejos = G["mercado"]["dist_vecinos_10"].to_numpy()[prueba]
umbral = np.nanquantile(lejos, 0.8)
escasos = lejos >= umbral
print(f"Pocos comparables: los 10 vecinos del mismo tipo a ≥{umbral:.1f} km en mediana · {mil(escasos.sum())} avisos de prueba")
comparar(escasos).style.format(FORMATO)

Pocos comparables: los 10 vecinos del mismo tipo a ≥0.3 km en mediana · 4.907 avisos de prueba


,MdAPE,±10%,±20%,avisos,frente al actual,IC 95%
Actual (base + mercado + servicios + texto),"13,8%","37,9%","65,0%",4.907,"+0,00 pts",—
Actual + censo DANE,"14,0%","37,7%","65,1%",4.907,"+0,25 pts","-0,07 pts a +0,56 pts"
Actual + capas IDESC,"13,8%","37,8%","64,6%",4.907,"+0,00 pts","-0,30 pts a +0,32 pts"
Actual + censo + IDESC,"14,0%","37,6%","65,1%",4.907,"+0,19 pts","-0,15 pts a +0,58 pts"
Censo en lugar de OSM,"13,9%","37,8%","65,0%",4.907,"+0,09 pts","-0,24 pts a +0,43 pts"


### ¿Y un modelo solo para Cali?

Las capas de la IDESC solo existen para el 8% de los avisos; un modelo nacional podría no aprovecharlas. Aquí,
el mismo LightGBM entrenado **solo con avisos de Cali** (con todas las variables), frente al nacional, en los
mismos avisos de prueba de Cali.

In [6]:
Xc = VARIANTES["Actual + censo + IDESC"]
aj_c, va_c, te_c = ajuste[es_cali[ajuste]], validacion[es_cali[validacion]], prueba[cali_te]
p_solo = mp.lightgbm(Xc, TE, y, aj_c, va_c, te_c)
e_solo = mp.error(p_solo, y[te_c])
pd.DataFrame({
    "Nacional, actual": mp.metricas(errores["Actual (base + mercado + servicios + texto)"][cali_te]),
    "Nacional, + censo + IDESC": mp.metricas(errores["Actual + censo + IDESC"][cali_te]),
    "Solo Cali, + censo + IDESC": mp.metricas(e_solo),
}).T.style.format(FORMATO)

,MdAPE,±10%,±20%,avisos
"Nacional, actual","11,3%","45,4%","74,2%",1.741
"Nacional, + censo + IDESC","11,7%","43,3%","72,7%",1.741
"Solo Cali, + censo + IDESC","11,8%","43,3%","72,3%",1.741


## 3. Qué aporta cada variable nueva (en Cali)

Importancia por permutación en los avisos de prueba de Cali, con el modelo nacional que tiene todo.

In [7]:
_, modelo, Xp = mp.lightgbm(VARIANTES["Actual + censo + IDESC"], TE, y, ajuste, validacion, prueba, devolver_modelo=True)
Xm, ym = Xp[cali_te], y[prueba][cali_te]
base_err = np.median(mp.error(modelo.predict(Xm), ym))
limpio = lambda c: c.replace(":", "_").replace(" ", "_")
imp = {}
for fuente in ("censo", "cali"):
    for c in G[fuente].columns:
        Xq = Xm.copy()
        orden = rng.permutation(len(Xq))
        Xq[limpio(c)] = pd.Series(Xq[limpio(c)].to_numpy()[orden], index=Xq.index, dtype=Xq[limpio(c)].dtype)
        imp[(("DANE" if fuente == "censo" else "IDESC"), c)] = np.median(mp.error(modelo.predict(Xq), ym)) - base_err
for fuente in ("censo", "cali"):              # cada fuente completa
    Xq = Xm.copy()
    orden = rng.permutation(len(Xq))
    for c in map(limpio, G[fuente].columns):
        Xq[c] = pd.Series(Xq[c].to_numpy()[orden], index=Xq.index, dtype=Xq[c].dtype)
    imp[(("DANE" if fuente == "censo" else "IDESC"), "— toda la fuente —")] = np.median(mp.error(modelo.predict(Xq), ym)) - base_err
imp = pd.Series(imp).sort_values(ascending=False)
imp.map(lambda v: f"{v * 100:+.2f} pts".replace(".", ",")).to_frame("error adicional en Cali al desordenarla").head(20)

error adicional en Cali al desordenarla
DANE  — toda la fuente —                                       +0,57 pts
      censo_pct_superior                                       +0,29 pts
IDESC cali_dist_via_colectora_km                               +0,26 pts
DANE  censo_pct_posgrado                                       +0,16 pts
IDESC cali_dist_colegio_km                                     +0,15 pts
      cali_ruido_dia_db                                        +0,14 pts
DANE  censo_viviendas                                          +0,14 pts
IDESC — toda la fuente —                                       +0,11 pts
      cali_n_paradas_mio_500m                                  +0,05 pts
      cali_homicidios_500m_3a                                  +0,05 pts
DANE  censo_pct_desocupadas                                    +0,04 pts
IDESC cali_centralidad                                         +0,04 pts
      cali_densidad_arborea                                    +0,04 pts
DANE  censo_pct_gas                                            +0,01 pts
IDESC cali_dist_mio_estacion_km                                +0,01 pts
      cali_inundacion                                          -0,01 pts
      cali_barrio_precario                                     -0,01 pts
DANE  censo_personas_ha                                        -0,02 pts
      censo_estrato_medio                                      -0,05 pts
      censo_pct_internet                                       -0,05 pts

## 4. Validación temporal (avisos nuevos de octubre)

In [8]:
t_tr, t_te = mp.particion_temporal(a, grupo)
t_aj, t_va = mp.subparticion(t_tr, grupo)
Gt = mp.variables_nuevas(a, y, t_aj, np.concatenate([t_va, t_te]), grupo, grupos=("mercado", "servicios", "texto"))
Gt |= {"censo": G["censo"], "cali": G["cali"]}             # no usan precios: no dependen de la partición
act_t = pd.concat([X, Gt["mercado"], Gt["servicios"], Gt["texto"]], axis=1)
var_t = {"Actual": act_t, "Actual + censo + IDESC": pd.concat([act_t, Gt["censo"], Gt["cali"]], axis=1)}
cali_t = es_cali[t_te]
filas = {}
for k, Xb in var_t.items():
    e = mp.error(mp.lightgbm(Xb, TE, y, t_aj, t_va, t_te), y[t_te])
    filas[(k, "todo el país")] = mp.metricas(e)
    filas[(k, "Cali")] = mp.metricas(e[cali_t])
print(f"Prueba: {mil(len(t_te))} avisos publicados desde el 1-oct ({mil(cali_t.sum())} de Cali)")
pd.DataFrame(filas).T.style.format(FORMATO)

Prueba: 14.459 avisos publicados desde el 1-oct (1.081 de Cali)


## 5. Conclusiones (estado del 26-oct-2026)

| Variante | Todo el país | Cali | Pocos comparables cerca |
|---|---|---|---|
| Actual (base + mercado + servicios + texto) | **10,7%** | **11,3%** | **13,8%** |
| + censo DANE | 10,8% | 11,4% | 14,0% |
| + capas IDESC | 10,8% | 11,6% | 13,8% |
| + censo + IDESC | 10,7% | 11,7% | 14,0% |
| Censo en lugar de OSM | 10,7% | 11,4% | 13,9% |
| Modelo solo de Cali (con todo) | — | 11,8% | — |

Ninguna diferencia es real: todos los intervalos de confianza incluyen el cero. En avisos nuevos de octubre,
Cali pasa de 10,7% a 10,5% con todo, dentro del ruido.

1. **Las fuentes oficiales no mejoran la estimación del precio.** El modelo sí las usa (desordenar el censo
   sube el error en Cali 0,77 puntos), pero sin ellas otras variables cubren lo mismo.
2. **La razón más probable:** el precio de los avisos vecinos ya incorpora la seguridad, el nivel educativo,
   el ruido o la cercanía al MIO. El mercado ya les puso precio, y el modelo lo lee de los comparables.
3. **Tampoco ayudan donde hay pocos comparables** (13,8% frente a 13,8–14,0%), ni con un modelo solo para
   Cali (11,8%, peor que el nacional: le faltan datos).
4. **El censo podría reemplazar a OpenStreetMap** sin perder nada (10,7% en ambos casos): es una opción si
   algún día OSM se vuelve difícil de mantener.

**Decisión:** el modelo de precios se queda con la configuración actual. El censo y las capas de la IDESC
**no** entran.

**Dónde sí valen la pena estas fuentes:** para **describir la zona**, no para estimar el precio.
- En el **informe de precio para captar** (producto 1 de E7): homicidios a 500 m, ruido, distancia al MIO,
  amenaza de inundación, nivel educativo y estrato del censo. Un agente puede explicarle al propietario
  *por qué* su zona vale lo que vale.
- En la **alerta**, como advertencia: "⚠️ zona de amenaza de inundación alta" o "barrio precario".
- En **ciudades nuevas** (E6), donde aún no haya suficientes avisos para comparar.